In [27]:
import torch

inputs = torch.tensor(
    [
        [0.43, 0.15, 0.89],  # Your     (x^1)
        [0.55, 0.87, 0.66],  # journey  (x^2)
        [0.57, 0.85, 0.64],  # starts   (x^3)
        [0.22, 0.58, 0.33],  # with     (x^4)
        [0.77, 0.25, 0.10],  # one      (x^5)
        [0.05, 0.80, 0.55],
    ]  # step     (x^6)
)

In [28]:
x_2 = inputs[1]
# inputs ko second vector (A) select gareko

d_in = inputs.shape[1]
# input vector ko dimension (B) nikaleko

d_out = 2
# output vector ko size set gareko

In [29]:
torch.manual_seed(123)
# random numbers same aauna ko lagi seed set gareko

W_query = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# query weight matrix banako (trainable jasto structure)
# requires_grad=False le training ma update hudaina

W_key = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# key weight matrix banako, attention compare garna use hunxa

W_value = torch.nn.Parameter(torch.rand(d_in, d_out), requires_grad=False)
# value weight matrix banako, final output represent garna use hunxa

In [30]:
query_2 = x_2 @ W_query
# x_2 lai query weight sanga multiply garera query vector banauxa

key_2 = x_2 @ W_key
# x_2 lai key weight sanga multiply garera key vector banauxa

value_2 = x_2 @ W_value
# x_2 lai value weight sanga multiply garera value vector banauxa
# actual information represent garxa

print(query_2)
# query vector print gareko

tensor([0.4306, 1.4551])


In [31]:
keys = inputs @ W_key
# inputs lai key weight sanga multiply garera sabai token ko key vector banauxa

values = inputs @ W_value
# inputs lai value weight sanga multiply garera sabai token ko value vector banauxa

print("keys.shape:", keys.shape)
# key matrix ko shape print gareko (kati token ra kati dimension cha bhanera dekhaunxa)

print("values.shape:", values.shape)
# value matrix ko shape print gareko (kati token ra kati dimension cha bhanera dekhaunxa)

keys.shape: torch.Size([6, 2])
values.shape: torch.Size([6, 2])


In [32]:
keys_2 = keys[1]
# keys ko second token select gareko

attn_score_22 = query_2.dot(keys_2)
# query ra key ko dot product garera attention score nikaleko
# yesle dherai similar xa vane high score dinxa

print(attn_score_22)
# final attention score print gareko

tensor(1.8524)


In [33]:
attn_scores_2 = query_2 @ keys.T
# query_2 lai sabai keys sanga multiply garera attention scores banauxa
# yesle x_2 le sabai tokens sanga kati relate garcha bhanera dekhaunxa

print(attn_scores_2)
# final attention score vector print gareko

tensor([1.2705, 1.8524, 1.8111, 1.0795, 0.5577, 1.5440])


In [34]:
d_k = keys.shape[-1]
# key vector ko dimension nikaleko (scale garna use hunxa)

attn_weights_2 = torch.softmax(attn_scores_2 / d_k**0.5, dim=-1)
# attention scores lai scale garera softmax lagako
# sqrt(d_k) le values stable banauxa (too large hunu bata bachauxa)
# dim=-1 le last dimension (sab tokens) ma probability banauxa

print(attn_weights_2)
# final attention weights print gareko
# yesle kun token lai kati focus garne bhanera dekhaunxa

tensor([0.1500, 0.2264, 0.2199, 0.1311, 0.0906, 0.1820])


In [35]:
context_vec_2 = attn_weights_2 @ values
# attention weights lai values sanga multiply garera final context vector banauxa
# yesle x_2 le sabai tokens bata kati information lina parxa bhanera combine garxa

print(context_vec_2)
# final context vector print gareko
# yesle attention ko final output dekhaunxa

tensor([0.3061, 0.8210])


In [36]:
import torch.nn as nn


class SelfAttention_v1(nn.Module):
    # self attention layer banako

    def __init__(self, d_in, d_out):
        super().__init__()
        # parent class initialize gareko

        self.W_query = nn.Parameter(torch.rand(d_in, d_out))
        # query weight matrix banako (trainable parameter)

        self.W_key = nn.Parameter(torch.rand(d_in, d_out))
        # key weight matrix banako

        self.W_value = nn.Parameter(torch.rand(d_in, d_out))
        # value weight matrix banako

    def forward(self, x):
        keys = x @ self.W_key
        # input lai key weight sanga multiply garera keys banauxa

        queries = x @ self.W_query
        # input lai query weight sanga multiply garera queries banauxa

        values = x @ self.W_value
        # input lai value weight sanga multiply garera values banauxa

        attn_scores = queries @ keys.T  # omega
        # query ra key bich similarity nikalera attention score banauxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scale garera softmax lagako
        # dim=-1 le sabai tokens ma probability banauxa

        context_vec = attn_weights @ values
        # attention weight use garera final context vector banauxa

        return context_vec

In [37]:
torch.manual_seed(123)
# random values same auna ko lagi seed set gareko

sa_v1 = SelfAttention_v1(d_in, d_out)
# SelfAttention class ko object create gareko

print(sa_v1(inputs))
# inputs lai self-attention layer ma pass garera output print gareko

tensor([[0.2996, 0.8053],
        [0.3061, 0.8210],
        [0.3058, 0.8203],
        [0.2948, 0.7939],
        [0.2927, 0.7891],
        [0.2990, 0.8040]], grad_fn=<MmBackward0>)


In [38]:
class SelfAttention_v2(nn.Module):
    # improved self-attention version

    def __init__(self, d_in, d_out, qkv_bias=False):
        super().__init__()

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera query banauxa (weight + optional bias)

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera key banauxa

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # linear layer use garera value banauxa

    def forward(self, x):

        keys = self.W_key(x)
        # input lai key linear layer bata pass garera keys banauxa

        queries = self.W_query(x)
        # input lai query linear layer bata pass garera queries banauxa

        values = self.W_value(x)
        # input lai value linear layer bata pass garera values banauxa

        attn_scores = queries @ keys.T
        # query ra key ko dot product garera attention score nikalxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scale garera softmax lagauxa
        # dim=-1 le last dimension ma probability banauxa

        context_vec = attn_weights @ values
        # attention weights use garera final context vector banauxa

        return context_vec

In [39]:
torch.manual_seed(789)
# random initialization same auna ko lagi seed set gareko

sa_v2 = SelfAttention_v2(d_in, d_out)
# SelfAttention_v2 class ko object create gareko (linear layers use garera)

print(sa_v2(inputs))
# inputs lai self-attention v2 ma pass garera output print gareko

tensor([[-0.0739,  0.0713],
        [-0.0748,  0.0703],
        [-0.0749,  0.0702],
        [-0.0760,  0.0685],
        [-0.0763,  0.0679],
        [-0.0754,  0.0693]], grad_fn=<MmBackward0>)


#causal attention

In [40]:
queries = sa_v2.W_query(inputs)  # A
keys = sa_v2.W_key(inputs)
attn_scores = queries @ keys.T
attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=1)
print(attn_weights)

tensor([[0.1921, 0.1646, 0.1652, 0.1550, 0.1721, 0.1510],
        [0.2041, 0.1659, 0.1662, 0.1496, 0.1665, 0.1477],
        [0.2036, 0.1659, 0.1662, 0.1498, 0.1664, 0.1480],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.1661, 0.1564],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.1585],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [41]:
context_length = attn_scores.shape[0]
# attention score matrix ko size bata sequence length nikaleko

mask_simple = torch.tril(torch.ones(context_length, context_length))
# lower triangular matrix banako (future tokens mask garna)
# tril le diagonal samma ko lower part 1 banauxa, mathi ko 0 hunxa

print(mask_simple)
# mask matrix print gareko
# yesle future token attention block garna use hunxa

tensor([[1., 0., 0., 0., 0., 0.],
        [1., 1., 0., 0., 0., 0.],
        [1., 1., 1., 0., 0., 0.],
        [1., 1., 1., 1., 0., 0.],
        [1., 1., 1., 1., 1., 0.],
        [1., 1., 1., 1., 1., 1.]])


In [42]:
masked_simple = attn_weights * mask_simple
# attention weights lai mask sanga multiply gareko
# future tokens (upper triangle) lai 0 banauxa
# yesle model le future information herna sakdaina

print(masked_simple)

tensor([[0.1921, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2041, 0.1659, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.2036, 0.1659, 0.1662, 0.0000, 0.0000, 0.0000],
        [0.1869, 0.1667, 0.1668, 0.1571, 0.0000, 0.0000],
        [0.1830, 0.1669, 0.1670, 0.1588, 0.1658, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<MulBackward0>)


In [43]:
row_sums = masked_simple.sum(dim=1, keepdim=True)
# each row ko sum nikaleko
# keepdim=True le shape same rakxa (broadcasting ko lagi)

masked_simple_norm = masked_simple / row_sums
# mask gareko attention weights lai normalize gareko
# each row sum = 1 banauxa (probability jasto)

print(masked_simple_norm)
# final normalized masked attention weights print gareko

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<DivBackward0>)


In [44]:
mask = torch.triu(torch.ones(context_length, context_length), diagonal=1)
# upper triangular mask banako (future tokens ko part)
# diagonal=1 le diagonal mathi ko sabai elements 1 banauxa

masked = attn_scores.masked_fill(mask.bool(), -torch.inf)
# mask bhayeko position haru ma -inf rakheko
# yesle softmax pachi tyo values 0 banauxa (completely ignore hunxa)

print(masked)

tensor([[0.2899,   -inf,   -inf,   -inf,   -inf,   -inf],
        [0.4656, 0.1723,   -inf,   -inf,   -inf,   -inf],
        [0.4594, 0.1703, 0.1731,   -inf,   -inf,   -inf],
        [0.2642, 0.1024, 0.1036, 0.0186,   -inf,   -inf],
        [0.2183, 0.0874, 0.0882, 0.0177, 0.0786,   -inf],
        [0.3408, 0.1270, 0.1290, 0.0198, 0.1290, 0.0078]],
       grad_fn=<MaskedFillBackward0>)


In [45]:
attn_weights = torch.softmax(masked / keys.shape[-1] ** 0.5, dim=1)
# sqrt(d_k) le values stable banauxa

print(attn_weights)

tensor([[1.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.5517, 0.4483, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.3800, 0.3097, 0.3103, 0.0000, 0.0000, 0.0000],
        [0.2758, 0.2460, 0.2462, 0.2319, 0.0000, 0.0000],
        [0.2175, 0.1983, 0.1984, 0.1888, 0.1971, 0.0000],
        [0.1935, 0.1663, 0.1666, 0.1542, 0.1666, 0.1529]],
       grad_fn=<SoftmaxBackward0>)


In [46]:
torch.manual_seed(123)
# random behavior same auna ko lagi seed set gareko

dropout = torch.nn.Dropout(0.5)  # A
# dropout layer banako (0.5 probability le elements drop garxa)
# training ma overfitting reduce garna use hunxa

example = torch.ones(6, 6)  # B
# 6x6 matrix banako sabai values 1 banayera

print(dropout(example))

tensor([[2., 2., 0., 2., 2., 0.],
        [0., 0., 0., 2., 0., 2.],
        [2., 2., 2., 2., 0., 2.],
        [0., 2., 2., 0., 0., 2.],
        [0., 2., 0., 2., 0., 2.],
        [0., 2., 2., 2., 2., 0.]])


In [47]:
torch.manual_seed(123)

print(dropout(attn_weights))
# overfitting reduce garna help garxa

tensor([[2.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000],
        [0.7599, 0.6194, 0.6206, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.4921, 0.4925, 0.0000, 0.0000, 0.0000],
        [0.0000, 0.3966, 0.0000, 0.3775, 0.0000, 0.0000],
        [0.0000, 0.3327, 0.3331, 0.3084, 0.3331, 0.0000]],
       grad_fn=<MulBackward0>)


In [48]:
batch = torch.stack((inputs, inputs), dim=0)
# same inputs lai batch ma combine gareko

print(batch.shape)
# yesle batch size, tokens, embedding dim dekhaunxa

torch.Size([2, 6, 3])


In [49]:
class CausalAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, qkv_bias=False):
        super().__init__()

        self.d_out = d_out

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        self.dropout = nn.Dropout(dropout)  # New
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # upper triangular mask banako
        # future tokens lai block garna use hunxa
        # register_buffer le model sanga save hunxa tara train hudaina

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, tokens, embedding dimension nikaleko

        keys = self.W_key(x)
        # input lai key space ma transform gareko

        queries = self.W_query(x)
        # input lai query space ma transform gareko

        values = self.W_value(x)
        # input lai value space ma transform gareko

        attn_scores = queries @ keys.transpose(1, 2)
        # batch-wise attention score nikaleko
        # transpose(1,2) le token dimension swap garxa

        attn_scores.masked_fill_(self.mask.bool()[:num_tokens, :num_tokens], -torch.inf)
        # future tokens ko attention -inf banako
        # softmax pachi tyo 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax lagako

        attn_weights = self.dropout(attn_weights)
        # dropout apply gareko (randomly kehi attention drop hunxa)

        context_vec = attn_weights @ values
        # final context vector banako (weighted sum)

        return context_vec

In [50]:
torch.manual_seed(123)

context_length = batch.shape[1]
# sequence length (token count per sample) nikaleko

ca = CausalAttention(d_in, d_out, context_length, 0.0)
# causal attention model create gareko
# 0.0 dropout means training ma pani dropout disable jasto hunxa

context_vecs = ca(batch)
# batch input lai causal attention layer ma pass gareko
# context-aware representation banauxa

print("context_vecs.shape:", context_vecs.shape)

context_vecs.shape: torch.Size([2, 6, 2])


#Multi head attention

In [51]:
class MultiHeadAttentionWrapper(nn.Module):
    # multi-head attention wrapper banako
    # same input lai multiple heads ma process garxa

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        self.heads = nn.ModuleList(
            [
                CausalAttention(d_in, d_out, context_length, dropout, qkv_bias)
                # each head ma separate causal attention layer banako
                for _ in range(num_heads)
            ]
        )

    def forward(self, x):
        return torch.cat([head(x) for head in self.heads], dim=-1)
        # sabai heads ko output nikalera concatenate gareko
        # dim=-1 le feature dimension ma join garxa
        # yesle multi-head representation banauxa

In [52]:
torch.manual_seed(123)
# random values same auna ko lagi seed set gareko

context_length = batch.shape[1]
# batch ko each sequence ma kati tokens cha bhanera nikaleko

d_in, d_out = 3, 2
# input dimension ra output dimension set gareko

mha = MultiHeadAttentionWrapper(d_in, d_out, context_length, 0.0, num_heads=2)
# multi-head attention model banako
# 2 heads use garera parallel attention compute garxa

context_vecs = mha(batch)
# batch input lai multi-head attention layer ma pass gareko
# each head le different context representation banauxa

print(context_vecs)
# final multi-head attention output print gareko

print("context_vecs.shape:", context_vecs.shape)
# output ko shape print gareko
# (batch_size, tokens, d_out * num_heads) hunxa

tensor([[[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]],

        [[-0.4519,  0.2216,  0.4772,  0.1063],
         [-0.5874,  0.0058,  0.5891,  0.3257],
         [-0.6300, -0.0632,  0.6202,  0.3860],
         [-0.5675, -0.0843,  0.5478,  0.3589],
         [-0.5526, -0.0981,  0.5321,  0.3428],
         [-0.5299, -0.1081,  0.5077,  0.3493]]], grad_fn=<CatBackward0>)
context_vecs.shape: torch.Size([2, 6, 4])


#multi head attention with weight split 
#Multi-Head Attention means the model uses multiple sets of Q, K, V weights instead of just one.

In [ ]:
class MultiHeadAttention(nn.Module):

    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"
        # d_out lai num_heads le divide garna milnu parxa
        # each head ko equal dimension ko lagi

        self.d_out = d_out

        self.num_heads = num_heads

        self.head_dim = d_out // num_heads

        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        # query projection layer banako

        self.W_key = nn.Linear(d_in, d_out, bias=qkv_bias)
        # key projection layer banako

        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)
        # value projection layer banako

        self.out_proj = nn.Linear(d_out, d_out)
        # sabai heads combine pachi final projection garna use hunxa

        self.dropout = nn.Dropout(dropout)
        # dropout layer banako (overfitting reduce garna)

        self.register_buffer(
            "mask", torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )
        # causal mask banako
        # future tokens block garna use hunxa

    def forward(self, x):

        b, num_tokens, d_in = x.shape
        # batch size, token count, input dimension nikaleko

        keys = self.W_key(x)
        # input lai key vectors ma transform gareko

        queries = self.W_query(x)
        # input lai query vectors ma transform gareko

        values = self.W_value(x)
        # input lai value vectors ma transform gareko

        # (b, num_tokens, d_out)
        # -> (b, num_tokens, num_heads, head_dim)

        keys = keys.view(b, num_tokens, self.num_heads, self.head_dim)
        # keys lai multiple heads ma split gareko

        values = values.view(b, num_tokens, self.num_heads, self.head_dim)
        # values lai multiple heads ma split gareko

        queries = queries.view(b, num_tokens, self.num_heads, self.head_dim)
        # queries lai multiple heads ma split gareko

        # (b, num_tokens, num_heads, head_dim)
        # -> (b, num_heads, num_tokens, head_dim)

        keys = keys.transpose(1, 2)
        # token ra head dimension swap gareko

        queries = queries.transpose(1, 2)
        # queries ko dimensions swap gareko

        values = values.transpose(1, 2)
        # values ko dimensions swap gareko

        attn_scores = queries @ keys.transpose(2, 3)
        # each head ko query ra key bich attention score nikaleko

        mask_bool = self.mask.bool()[:num_tokens, :num_tokens]
        # required token size anusar mask select gareko

        attn_scores.masked_fill_(mask_bool, -torch.inf)
        # future token positions lai -inf banako
        # softmax pachi probability 0 hunxa

        attn_weights = torch.softmax(attn_scores / keys.shape[-1] ** 0.5, dim=-1)
        # scaled softmax attention apply gareko

        attn_weights = self.dropout(attn_weights)
        # attention weights ma dropout apply gareko

        context_vec = (attn_weights @ values).transpose(1, 2)
        # weighted sum garera context vector banako
        # transpose garera original order ma lyako

        context_vec = context_vec.contiguous().view(b, num_tokens, self.d_out)
        # sabai heads combine garera reshape gareko

        context_vec = self.out_proj(context_vec)
        # final linear projection apply gareko

        return context_vec
        # final multi-head attention output return gareko

In [ ]:
a = torch.tensor(
    [
        [
            [
                [0.2745, 0.6584, 0.2775, 0.8573],  # A head 1
                [0.8993, 0.0390, 0.9268, 0.7388],
                [0.7179, 0.7058, 0.9156, 0.4340],
            ],
            [
                [0.0772, 0.3565, 0.1479, 0.5331],
                [0.4066, 0.2318, 0.4545, 0.9737],  # B head 2
                [0.4606, 0.5159, 0.4220, 0.5786],
            ],
        ]
    ]
)

# 4D tensor banako
# shape: (1, 2, 3, 4)

# 1 = batch size
# 2 = number of heads
# 3 = number of tokens
# 4 = head dimension

# yo tensor multi-head attention ko example representation ho

In [ ]:
print(a @ a.transpose(2, 3))

# last 2 dimensions transpose gareko
# (tokens, head_dim) -> (head_dim, tokens)

# @ operator le matrix multiplication gareko
# each head ko token bich similarity score nikalxa

# yo multi-head attention ko attention score calculation jastai ho

tensor([[[[1.3208, 1.1631, 1.2879],
          [1.1631, 2.2150, 1.8424],
          [1.2879, 1.8424, 2.0402]],

         [[0.4391, 0.7003, 0.5903],
          [0.7003, 1.3737, 1.0620],
          [0.5903, 1.0620, 0.9912]]]])


In [ ]:
first_head = a[0, 0, :, :]
# batch 0 ko first attention head select gareko
# shape: (tokens, head_dim)

first_res = first_head @ first_head.T
# first head vitra token-token similarity (attention score) nikaleko
# T le transpose garera dot product gareko

print("First head:\n", first_res)
# first head ko result print gareko

second_head = a[0, 1, :, :]
# batch 0 ko second attention head select gareko

second_res = second_head @ second_head.T
# second head ko token-token similarity calculate gareko

print("\nSecond head:\n", second_res)
# second head ko attention score matrix print gareko

First head:
 tensor([[1.3208, 1.1631, 1.2879],
        [1.1631, 2.2150, 1.8424],
        [1.2879, 1.8424, 2.0402]])

Second head:
 tensor([[0.4391, 0.7003, 0.5903],
        [0.7003, 1.3737, 1.0620],
        [0.5903, 1.0620, 0.9912]])


In [ ]:
torch.manual_seed(123)

batch_size, context_length, d_in = batch.shape
# batch ko shape unpack gareko
# batch_size = number of samples
# context_length = number of tokens
# d_in = input embedding dimension

d_out = 2
# total output dimension set gareko

mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
# multi-head attention model banako
# 2 heads use garera parallel attention compute garxa

context_vecs = mha(batch)
# batch input lai multi-head attention layer ma pass gareko
# each head le different representation learn garxa

print(context_vecs)
# final output (context vectors) print gareko

print("context_vecs.shape:", context_vecs.shape)
# output ko shape print gareko
# expected shape: (batch_size, context_length, d_out)
# because heads combine bhaisakepachi final projection lagxa

tensor([[[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]],

        [[0.3190, 0.4858],
         [0.2943, 0.3897],
         [0.2856, 0.3593],
         [0.2693, 0.3873],
         [0.2639, 0.3928],
         [0.2575, 0.4028]]], grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 6, 2])
